In [9]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.graph.message import MessagesState
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv(override=True)

model = ChatOpenAI(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": { "type": "disabled" }
    }
)
class OverAllState(MessagesState):
    username: str
    output: str

def node_a(state: OverAllState) -> dict:
    return {
        "messages": [HumanMessage("你好，我是" + state["username"])]
    }

def llm_node(state: OverAllState) -> dict:
    res = model.invoke(state["messages"])
    return {
        "messages": [res],
        "output": res.content
    }

# 3、构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)
builder.add_edge(START,"node_a")
builder.add_edge("node_a","llm_node")
builder.add_edge("llm_node",END)

graph = builder.compile()

# 运行图

result = graph.invoke({ "username": "你好老王" })

rprint(result)


{
    'messages': [
        HumanMessage(
            content='你好，我是你好老王',
            additional_kwargs={},
            response_metadata={},
            id='9ae04ef4-bd57-483d-8b9f-405c0fbe6ca3'
        ),
        AIMessage(
            content='你好，老王！😄 有什么我可以帮你的吗？无论是闲聊、解答问题，还是需要一些建议，我都在这里！',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 28,
                    'prompt_tokens': 9,
                    'total_tokens': 37,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-72b5a98f-7616-9bca-915c-2da185bbb11b',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a083fb-a1d1-7730-9937-dcf63e5ff6f0-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 9,
                'output_tokens': 28,
                'total_tokens': 37,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ],
    'username': '你好老王',
    'output': '你好，老王！😄 有什么我可以帮你的吗？无论是闲聊、解答问题，还是需要一些建议，我都在这里！'
}